### Imports 

In [ ]:
from pathlib import Path
import numpy as np
import pandas
import geopandas as gpd
import pandas as pd
import scipy
import re
import sys, pathlib, importlib

sys.path.append("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
import Robyn_river_floods




### Base paths and output path 

In [ ]:
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info")
output_path.mkdir(parents=True, exist_ok=True)
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
DATA_DIR = base_path / "Processed_data/direct_damages_fred"
basins_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_basins_100")

#### Read in forest cover stats

In [ ]:
# import the catchment forest area statistics cvs
upstream_basin_cover = Path("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/basin_aff_ffe_areas.csv")
upstream_basin_cover = pd.read_csv(upstream_basin_cover)
display(upstream_basin_cover.head())          # or df.head()

#### Read in peak flow reduction information

In [ ]:
# Load interpolated peak flow reduction data
peak_flow_catchment_coverage_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/interpolated_peak_flow_catchment_coverage.csv")
peak_flow_catchment_coverage = pandas.read_csv(peak_flow_catchment_coverage_path)
peak_flow_catchment_coverage

#### % Change in peak flow impact on return period

In [ ]:
# Top row outlines change in peak flow. Subsequent rows define the return period and what it becomes.
### Assume for return period of 2 or below, flood depth is 0, i.e. the infrastructure asset experiences no damage. 

""
"Need to add in 30% values - is there a way to interpolate this?"
""

CCRA_flow_reductions = pandas.DataFrame({
    'reduction_percent': [5, 10, 20, 40],
    #'rp2.0': [2.4, 3.2, 6.8, 196],
    #'rp2.3': [2.8, 3.7, 7.9, 169],
    'rp5.0': [6.5, 8.9, 19, 235],
    'rp10.0': [13,19,43,473],
    #'rp25.0': [35, 51, 123, 1330],
    'rp50.0': [72,107,268,2967],
    'rp100.0': [147,224,582, 6648],
    #'rp500.0': [772,1229,3441,43094],
    #'rp1000.0': [1571,2544,7345,95943],
})

# proportion of baseline flow
CCRA_flow_reductions['flow'] = (1 - CCRA_flow_reductions.reduction_percent / 100)

# The below code is used to interpolate based on the current return periods (2,12,50,100,500,1000) to define a new return period (20)
## interpolate RP20 values
known_rp_cols = [rp_col for rp_col in CCRA_flow_reductions.columns if "rp" in rp_col]
known_rps = [float(rp.replace("rp","")) for rp in known_rp_cols]
flows = CCRA_flow_reductions['flow'].values

interpolator = scipy.interpolate.RegularGridInterpolator(
    (known_rps, flows),
    CCRA_flow_reductions[known_rp_cols].values.T,
    method='cubic'
)

rps_new = [[20.0]]
CCRA_flow_reductions['rp20.0'] = interpolator((rps_new, [flows]))[0]

out_rps = sorted([20.0] + known_rps)
out_rp_cols = [f"rp{rp}" for rp in out_rps]

CCRA_flow_reductions_interpolated = CCRA_flow_reductions[['flow'] + out_rp_cols].copy()

flow07_row_values = interpolator(([out_rps], [[0.7]]))[0]
flow07_row = pandas.DataFrame(data=[[0.7] + list(flow07_row_values)], columns=['flow']+out_rp_cols)

CCRA_flow_reductions_interpolated = (
    pandas.concat([CCRA_flow_reductions_interpolated, flow07_row])
    .sort_values("flow", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
def peak_flow_reduction(forest_perc, lookup):
    data = lookup.catchment_forest_percentage
    rp_cols, rps = get_rp_cols(lookup)
    interpolator = scipy.interpolate.RegularGridInterpolator(
        (rps, data),
        lookup[rp_cols].values.T,
        method='linear'
    )
    vals = interpolator(([rps], [forest_perc]))[0]
    return pandas.DataFrame({
        'catchment_forest_percentage': [forest_perc],
        'rp5.0': vals[0],
        'rp100.0': vals[1],
    })

peak_flow_reduction(7, peak_flow_catchment_coverage)

In [ ]:
# Define the known return periods (the ones you have columns for)
rp_known = [20, 50, 100, 200, 500, 1500]

# Define the full range to interpolate over (here every integer from 20 to 1500)
rp_full = np.arange(20, 1501)


In [ ]:
# for change in land cover what is the change in RP - should be bigger or the same if no forest is added
# 
display(CCRA_flow_reductions_interpolated)

### Calculate return period change for each catchment forest coverage 

In [ ]:
def interpolate_flow_reductions(current_cover_perc, future_cover_perc, flow_reductions_df, return_period):
    """
    Interpolates peak flow reductions based on current and future forest cover percentages.
    
    Parameters:
        current_cover_perc (float): The current forest cover percentage.
        future_cover_perc (float): The future forest cover percentage (e.g., after reforestation).
        flow_reductions_df (pd.DataFrame): DataFrame containing forest cover percentages and flow reductions.
        return_period (str): The column name in the DataFrame for the return period (e.g., 'rp5.0').
    
    Returns:
        tuple: (current_flow_reduction, future_flow_reduction) interpolated for the given return period.
    """
    # Ensure percentages are within bounds (of 0-100)
    current_cover_perc = max(0, min(100, current_cover_perc))
    future_cover_perc = max(0, min(100, future_cover_perc))
    
    # Interpolate flow reductions for the current and future percentages
    current_flow_reduction = np.interp(
        current_cover_perc,
        flow_reductions_df['catchment_forest_percentage'],
        flow_reductions_df[return_period]
    )
    
    future_flow_reduction = np.interp(
        future_cover_perc,
        flow_reductions_df['catchment_forest_percentage'],
        flow_reductions_df[return_period]
    )
    
    return current_flow_reduction, future_flow_reduction

upstream_basin_cover = upstream_basin_cover.copy()


# Define return periods to analyze
return_periods = ['rp5.0', 'rp100.0']

# Loop through each return period and calculate reductions
for return_period in return_periods:
    # 1) Interpolate current & future flow reductions
    upstream_basin_cover[f'current_flow_reduction_{return_period}'], upstream_basin_cover[f'future_flow_reduction_{return_period}'] = zip(*upstream_basin_cover.apply(
        lambda row: interpolate_flow_reductions(
            current_cover_perc=row['existing_forest_pct'],
            future_cover_perc=row['total_future_forest_pct'],
            flow_reductions_df=peak_flow_catchment_coverage,
            return_period=return_period
        ),
        axis=1
    ))
    
    # 2) Compute the ratio-based columns
    upstream_basin_cover[f'future_ratio_{return_period}'] = 100 - upstream_basin_cover[f'future_flow_reduction_{return_period}']
    upstream_basin_cover[f'current_ratio_{return_period}'] = 100 - upstream_basin_cover[f'current_flow_reduction_{return_period}']
    
    upstream_basin_cover[f'change_in_ratio_{return_period}'] = (
        upstream_basin_cover[f'current_ratio_{return_period}'] 
        - upstream_basin_cover[f'future_ratio_{return_period}']
    )
    
    upstream_basin_cover[f'future_reduction_proportion_{return_period}'] = (
        upstream_basin_cover[f'change_in_ratio_{return_period}'] 
        / upstream_basin_cover[f'current_ratio_{return_period}']
    ) * 100

# Now do rounding
columns_to_round = [col for col in upstream_basin_cover.columns if 'flow_reduction' in col or 'reduction_difference' in col]
columns_to_round += [col for col in upstream_basin_cover.columns if 'ratio' in col or 'future_reduction_proportion' in col]
upstream_basin_cover[columns_to_round] = upstream_basin_cover[columns_to_round].round(2)

# Debug
display(
    upstream_basin_cover[
        [
            'basin_file', 
            'existing_forest_pct', 
            'total_future_forest_pct'
        ] 
        + columns_to_round
    ].head()
)

# write ONE combined CSV with all columns
combined_out = output_path / "peak_flow_reductions_by_basin.csv"
upstream_basin_cover.to_csv(combined_out, index=False)
print(f"Wrote {combined_out}")

In [ ]:
# example data frame of pre-calculated total damages in each upstream basin for a set of return periods
damages = pandas.DataFrame(
    data={
        "basin_file": [123, 124, 125],
        "rp0.001": [0,0,0],
        "rp2.0": [0,0,0],
        "rp20.0": [10,20,30]
    }
).set_index("basin_file")

# Define output return periods
rps_to_calculate = [2.0, 5.0, 10.0]

# Call the calculate_rp_maps function
interpolated_damages = Robyn_river_floods.interpolate_rp_damages(rps_to_calculate, damages)
interpolated_damages

In [ ]:
Robyn_river_floods.peak_flow_reduction(forest_percentage_change=[0, 10, 100])

In [ ]:
rp_cols = ["rp5.0","rp10.0","rp20.0","rp50.0","rp100.0"]
catchment_peak_flow_reduction = Robyn_river_floods.peak_flow_reduction(upstream_basin_cover.afforestable_pct) \
    [rp_cols] \
    .rename(columns={rp_col: f"pfr_{rp_col}" for rp_col in rp_cols})

catchments_with_rp_change = upstream_basin_cover.join(catchment_peak_flow_reduction)
catchments_with_rp_change

for rp_col in rp_cols:
    rp_change = Robyn_river_floods.rp_change_given_flow_reduction(reduction_percent=catchments_with_rp_change[f"pfr_{rp_col}"], interp_rp=float(rp_col.replace("rp", ""))) \
        [[rp_col]] \
        .rename(columns={rp_col: f"future_{rp_col}"})
    catchments_with_rp_change = catchments_with_rp_change.join(rp_change)

catchments_with_rp_change

In [ ]:
# read split damages (done)
# read flooded points snapped to river network  from merged_fid (done)
# for each flooded point
  # find the set of damaged (split) assets at that point (cell_index_2_x/y) or flood_i/j - few rows dataframe subset of split damages
  # find the catchment row at the related river network point (dem_i/j) - one row dataframe subset of catchments_with_rp_change
  # call adapted version of calculate_future_sector_damages  (without the hybas or variant for loops)

### Read in split damages

In [ ]:
# Example damage file
def read_damage_file(fname):
    example_damage = pd.read_parquet(fname) 
    example_damage.head()
    to_drop_col_names = [
        col for col in example_damage.columns 
        if col.startswith("coastal") 
        or col.startswith("surface") 
        or (col.startswith("fluvial") and "baseline" not in col)
    ]
    to_drop_cell_index_cols = [
        col for col in example_damage.columns 
        if col.startswith("cell_index") 
        and "2" not in col
    ]
    example_damage = example_damage.drop(columns=to_drop_col_names+to_drop_cell_index_cols)
    
    return example_damage

example_damage = read_damage_file(DATA_DIR / "damages/rail_edges_direct_damages_parameter_set_2.parquet").query('fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None > 0')
example_damage.columns, example_damage.shape

example_damage.head()

### Read flooded points snapped to river network from merged_fid

In [ ]:
flooded_points_snapped = pd.read_parquet("merged_fid_subset.parquet")

In [ ]:
flooded_point = flooded_points_snapped.query("flood_i == 4745 and flood_j == 1281").iloc[0]

### Find the damages

In [ ]:
def calculate_damages_at_point(flooded_point, all_damages, catchments):
    flood_i = flooded_point.flood_i
    flood_j = flooded_point.flood_j
    # use this to find subset of damages that have that flood_i and flood_j as their cell_index_2x and y
    damage_subset = all_damages.query(f"cell_index_2_x == {flood_i} and cell_index_2_y == {flood_j}")
    # use the flood_i and j to find the rail edges, then the dem ones to find the catchment. 
    dem_i = flooded_point.dem_i
    dem_j = flooded_point.dem_j
    catchment = catchments.query(f"dem_i_file == {dem_i} and dem_j_file == {dem_j}").drop_duplicates()
    
    
    damage_cols_to_rp = {    
        'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None': 'rp20.0',
        'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None': 'rp50.0',
        'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None': 'rp100.0',
        'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None': 'rp200.0',
        'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None': 'rp500.0',
        'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None': 'rp1500.0',
    }
    damage_subset_rps = damage_subset[damage_cols_to_rp.keys()].rename(columns=damage_cols_to_rp)
    damage_subset_rps['rp0.0001'] = 0.0
    damage_subset_rps['rp2.0'] = 0.0
    damage_subset_rps["rp1000000000.0"] = damage_subset_rps['rp1500.0']
    
    interpolated_baseline_damages = Robyn_river_floods.interpolate_rp_damages(
        [5.0, 10.0],
        damage_subset_rps
    )
    
    baseline_damages = damage_subset_rps.join(interpolated_baseline_damages)
    
    future_rps = [float(colname.replace("future_rp", "")) for colname in catchment.columns if "future_rp" in colname]
    future_rp_columns = [f"rp{rp}" for rp in future_rps]
    # get the *future* RP values corresponding to each
    current_to_future_rps = {
        rp: catchment[f"future_rp{float(rp)}"].iloc[0]
        for rp in future_rps
    }
    future_damages = baseline_damages[future_rp_columns].rename(columns={
        f"rp{current}": f"rp{future}"
        for current, future in current_to_future_rps.items()
    })
    extreme_damages = baseline_damages[["rp0.0001", "rp1000000000.0"]].copy()
    future_damages = future_damages.join(extreme_damages)
    future_damages_columns = sorted(
        future_damages.columns, 
        key=lambda c: float(c.replace("rp", ""))
    )
    future_damages = future_damages[future_damages_columns].copy()
    # at this point future damages contains columns for each of the adjusted future return periods
    # corresponding to the initial baseline set (e.g. rp6.4, rp12.0, rp24.1 corresponding to what was
    # rp5.0, rp10.0, rp20.0) - these could be used to calculate EAD directly, without the next bit
    # of interpolation.
    
    # calculate baseline EAD
    baseline_ead_values = Robyn_river_floods.calculate_ead(baseline_damages[future_rp_columns])  # based on RPs up to 100
    baseline_ead_colname = f"baseline__fluvial__ead" 
    
    interpolated_future_damages = Robyn_river_floods.interpolate_rp_damages(
        future_rps,
        future_damages
    )
    
    future_ead_values = Robyn_river_floods.calculate_ead(interpolated_future_damages)
    future_ead_colname = f"future__fluvial__ead"
    # now rename its *output* columns (one per rp_to_calculate)
    to_rename = {
        f"rp{rp}": f"future__fluvial__rp_{int(rp)}"
        for rp in future_rps
    }
    interpolated_future_damages.rename(columns=to_rename, inplace=True)
    # pick out the interpolated future rp damages
    interpolated_future_damages = interpolated_future_damages[to_rename.values()]
    # add future ead as calculated above
    interpolated_future_damages[future_ead_colname] = future_ead_values
    
    # rename from rpXX columns to baseline...
    to_rename = {
        f"rp{rp}": f"baseline__fluvial__rp_{int(rp)}"
        for rp in future_rps
    }
    baseline_damages_renamed = baseline_damages.rename(columns=to_rename)[to_rename.values()]
    baseline_damages_renamed[baseline_ead_colname] = baseline_ead_values
    
    # join the interpolated baseline rp damages
    interpolated_future_damages = pandas.concat([
        interpolated_future_damages,
        baseline_damages_renamed
    ], axis=1)
    
    interpolated_future_damages["edge_id"] = damage_subset["edge_id"]
    interpolated_future_damages["flood_i"] = flood_i
    interpolated_future_damages["flood_j"] = flood_j
    interpolated_future_damages["dem_i"] = dem_i
    interpolated_future_damages["dem_j"] = dem_j
    
    return interpolated_future_damages.copy()

calculate_damages_at_point(flooded_point, example_damage, catchments_with_rp_change)

In [ ]:
future_damage_dfs = []
for point in flooded_points_snapped.itertuples():
    point_damages = calculate_damages_at_point(point, example_damage, catchments_with_rp_change)
    future_damage_dfs.append(point_damages)

future_damage = pd.concat(future_damage_dfs, axis=0)

In [ ]:
damages_by_flooded_point = future_damage.query("baseline__fluvial__ead > 0")[
    ["flood_i","flood_j","dem_i","dem_j","baseline__fluvial__ead", "future__fluvial__ead"]
].groupby(["flood_i","flood_j"]).agg({
    "dem_i": "first",
    "dem_j":"first",
    "baseline__fluvial__ead": "sum",
    "future__fluvial__ead": "sum",
})

In [ ]:
future_damage.to_parquet("rail_edges_direct_damages_parameter_set_2__future.parquet")

In [ ]:
damages_by_flooded_point.reset_index().iloc[0]

In [ ]:
# find the catchment gpkg for dem_i dem_j
# read in the catchment polygon
# intersect catchment with landuse points, filtered to only the afforestable points
# catchment_aff_points_df["damage_reduction_proportion"] = flooded_point.damage_reduction / len(catchment_points_df)
# catchment_aff_points_df > save to parquet or append long list

In [ ]:
by_edge = future_damage.groupby("edge_id").sum()
by_edge["damage_reduction"] = by_edge.baseline__fluvial__ead - by_edge.future__fluvial__ead
by_edge["damage_reduction"].plot.bar()

In [ ]:
example_damage.query("cell_index_2_x == 3464 and cell_index_2_y == 527")

#### Trying to figure out linking avoided damages to land use

In [ ]:
future_damage = pd.read_parquet("rail_edges_direct_damages_parameter_set_2__future.parquet")
future_damage

In [ ]:
# Create new column
future_damage["avoided_damages_ead"] = (
    future_damage["baseline__fluvial__ead"] - future_damage["future__fluvial__ead"]
)

# Create new column
future_damage["avoided_damages_ead_pct"] = (
    future_damage["avoided_damages_ead"] / future_damage["baseline__fluvial__ead"]
)*100


# (optional) clip negative values to 0
# future_damage["avoided_damages_ead"] = future_damage["avoided_damages_ead"].clip(lower=0)

# Quick check
print(future_damage[["baseline__fluvial__ead", "future__fluvial__ead", "avoided_damages_ead"]].head())
future_damage

future_damage.to_csv("future_damage_with_avoided.csv", index=False)

In [ ]:
### Do it for one

### Figure out corresponding afforestable points to go with EADs 

In [ ]:
aff_pts = gpd.read_file("/Users/robynhaggis/Documents/Geospatial_analysis/upstream_catchment_info/aff_points_with_ffe.gpkg")

In [ ]:
# Extract i,j from the filename
dem_i, dem_j = 3807, 1851

def portion_damage_reduction(dem_i, dem_j, basin_damages, aff_pts):
    # read basin
    gpkg_path = Path(f"/Users/robynhaggis/Documents/Geospatial_analysis/upstream_basins_100/basin_{dem_i}_{dem_j}.gpkg")
    basin = gpd.read_file(gpkg_path)
    # filter damages
    total_avoided_in_basin = basin_damages["avoided_damages_ead"].sum()
    
    # points in basin
    pts_in_basin = gpd.sjoin(
        aff_pts,
        basin[["geometry"]],
        how="inner",
        predicate="within",
    ).drop(columns=["index_right"])
    
    afforestable_pts = pts_in_basin[pts_in_basin["afforestable"] > 0].copy()
    total_afforestable = afforestable_pts.afforestable.sum()
    
    if total_afforestable == 0:
        return afforestable_pts
    
    avoided_per_unit = total_avoided_in_basin / total_afforestable
    afforestable_pts["avoided_ead_portion"] = afforestable_pts["afforestable"] * avoided_per_unit
    
    return afforestable_pts

basin_damages = future_damage[(future_damage["dem_i"] == dem_i) & (future_damage["dem_j"] == dem_j)]
portion_damage_reduction(dem_i, dem_j, basin_damages, aff_pts)

In [ ]:
dfs = []
for (dem_i, dem_j), pixel_damages in future_damage.groupby(['dem_i','dem_j']):
    df = portion_damage_reduction(dem_i, dem_j, pixel_damages, aff_pts)
    dfs.append(df)
damage_reduction_long = pandas.concat(dfs)

In [ ]:
damage_reduction = damage_reduction_long.query('avoided_ead_portion > 0').groupby(['dem_i','dem_j']).agg({
    'dem_i':'first',
    'dem_j':'first',
    'afforestable':'first',
    'existing_forest':'first',
    'geometry':'first',
    'avoided_ead_portion':'sum',
}).reset_index(drop=True)

In [ ]:
gpd.GeoDataFrame(damage_reduction, crs=3448).to_file(base_path / "avoided_ead_portion.gpkg")